In [ ]:
# поместить подпись в asn1 файл
import asn1
from asn1 import Encoder, Decoder, Numbers

# запись в asn1 (Приложение Г методичесого пособия)
def Asn_1(A, B, p, q, r, s, x_Q, y_Q, x_P, y_P, file_name,data):
    encoder = asn1.Encoder()
    encoder.start()
    encoder.enter(Numbers.Sequence) # заголовок
    encoder.enter(Numbers.Set) # множество ключей, 1 задействован
    encoder.enter(Numbers.Sequence) # первый «ключ»
    encoder.write(b'\x80\x06\x07\x00', Numbers.OctetString) # идентификатор алгоритма (протокол подписи ГОСТ)
    encoder.enter(Numbers.Sequence) # значение открытого ключа
    encoder.write(int(x_Q), Numbers.Integer) # x-координата точки Q
    encoder.write(int(y_Q), Numbers.Integer) # y-координата точки Q
    encoder.leave()
    encoder.enter(Numbers.Sequence) # параметры криптосистемы
    encoder.enter(Numbers.Sequence) # параметры поля
    encoder.write(p, Numbers.Integer) # простое число p
    encoder.leave()
    encoder.enter(Numbers.Sequence) # параметры кривой
    encoder.write(A, Numbers.Integer) # коэффициент A уравнения кривой
    encoder.write(B, Numbers.Integer) # коэффициент B уравнения кривой
    encoder.leave()
    encoder.enter(Numbers.Sequence) # образующая группы точек кривой
    encoder.write(int(x_P), Numbers.Integer) # x-координата образующей точки P
    encoder.write(int(y_P), Numbers.Integer) # y-координата образующей точки P
    encoder.leave()
    encoder.write(q, Numbers.Integer) # порядок группы q
    encoder.leave()
    encoder.enter(Numbers.Sequence) # подпись сообщения
    encoder.write(int(r), Numbers.Integer) # число r
    encoder.write(int(s), Numbers.Integer) # число s
    encoder.leave()
    encoder.leave()
    encoder.leave()
    encoder.enter(Numbers.Sequence) # параметры файла, не используются
    encoder.leave()
    encoder.leave()
    with open(file_name, 'wb') as output_file:
        output_file.write(encoder.output())
        output_file.write(data)

In [ ]:
# достать подпись из asn1 файла
def get_header_length(data):
    first_tag_len = int(data[1]) # длина проверка
        ##print("DATA ", first_tag_len)
    if first_tag_len > 127:
        #print(data)
        count_length_bait = int(first_tag_len & 0x7F)
        header_len = data[2:2+count_length_bait]
        header_len = int.from_bytes(header_len, byteorder='big')
        #print("header_len = ",header_len)
        header_len +=4
        
            ##additional_bytes = data[1:1 + header_length]   # Дополнительные байты, содержащие длину заголовка
            ##header_length += len(additional_bytes)   # Вычислите длину заголовка, основываясь на количестве байт в дополнительных байтах
    else:
        header_len = int(first_tag_len & 0x7F)
        header_len += 2
            
    return header_len

def From_Asn_1(file_name):
    with open(file_name, "rb") as input_file:
        sign_data = input_file.read()
    asn1 = Decoder()
    asn1.start(sign_data)
    asn1.enter() 
    asn1.enter() 
    asn1.enter() 
    value = asn1.read() 
    asn1.enter() 
    x_Q = asn1.read()[1] 
    y_Q = asn1.read()[1] 
    asn1.leave()
    asn1.enter() 
    asn1.enter()
    p = asn1.read()[1] 
    asn1.leave() 
    asn1.enter() 
    A = asn1.read()[1] 
    B = asn1.read()[1]  
    asn1.leave()
    asn1.enter() 
    x_P = asn1.read()[1] 
    y_P = asn1.read()[1] 
    asn1.leave()
    q = asn1.read()[1] 
    asn1.leave()
    asn1.enter() 
    r = asn1.read()[1]
    s = asn1.read()[1] 
    asn1.leave()
    asn1.leave()
    asn1.leave()
    asn1.enter()
    asn1.leave()
    asn1.leave()
    
    header_len=get_header_length(sign_data)
    data = sign_data[header_len:len(sign_data)]
    
    return A, B, p, q, r, s, x_Q, y_Q, x_P, y_P,data


In [ ]:
#Протокол подписи согласно ГОСТ Р 34.10−2018
import os
from Cryptodome.Hash import SHA256, SHA512


def GOST_34_10_2018_sender(p,q,A,B,x_P,y_P,d,file_name,new_file_name):
    size=os.path.getsize(file_name)
    with open(file_name, 'rb') as file:
        data = file.read(size)
    hash_instance = SHA256.new()
    hash_instance.update(data)
    # Получение байтовой строки хеша
    hash_bytes = hash_instance.digest()
    print(f"Хэш = {hash_bytes}")
    hash_integer = int.from_bytes(hash_bytes, byteorder='big')
    e = hash_integer % q
    if(e==0):
        e=1
    K = GF(p)
    E = EllipticCurve(K, [A, B])
    P = E(x_P, y_P)
    Q = d*P
    #Multiplication_by_a_point(P,k_)## РЕАЛИЗОВАТЬ!!!
    while(1):
        k=randint(0, q)# можно попробовтаь реалтзовтаь умножение
        C = k * P 
        #r= C[0] % q
        r = mod(C[0], q)
        if r == 0:
            continue
        s=(r*d+k*e) % q
        if s!=0:
            break      
    binary_r = bin(r)[2:]  # Убираем префикс '0b'
    binary_s = bin(s)[2:]  # Убираем префикс '0b'
#     print("r =",binary_r)
#     print(" s=",binary_s)
    Asn_1(A, B, p, q, binary_r, binary_s, Q[0], Q[1], x_P, y_P, new_file_name,data)
    return 0



def GOST_34_10_2018_recipient(file_name,new_file):
    A, B, p, q, binary_r, binary_s, x_Q, y_Q, x_P, y_P, data = From_Asn_1(file_name)
    K = GF(p)
    E = EllipticCurve(K, [A, B])
    P = E(x_P, y_P)
    Q = E(x_Q,y_Q)
    r=int(str(binary_r), 2)
    s=int(str(binary_s), 2)
    if (r<0) or (r>q) or (s<0) or (s>q):
        print("THE SIGN IS INCORRECT")
        return 0
    hash_instance = SHA256.new()
    hash_instance.update(data)
    # Получение байтовой строки хеша
    hash_bytes = hash_instance.digest()
    hash_integer = int.from_bytes(hash_bytes, byteorder='big')
    e = hash_integer % q
    if(e==0):
        e=1
    v = inverse_mod(e, q)
    z_1 = (s*v) % q
    z_2= (-r*v) % q    
    C = z_1*P + z_2*Q
    R = mod(C[0], q)
    if(R==r):
        print("THE SIGN IS CORRECT")
        with open(new_file, "wb") as f:
            f.write(data)
        return A, B, p, q, r, s, x_Q, y_Q, x_P, y_P, data
    else:
        print("THE SIGN IS INCORRECT")
        return 0
    
    
    
    
#file_name = 'C:\\Users\\HONOR\\Desktop\\Учеба\\КМЗИ\\лаб 3\\x\\test.txt'   
#decode_file=file_name+'.decode'

def main():
    file_name = 'open_text' 
    decode_file=file_name+'.signed'
    while (1):
        print("1 - Creating a signature for a message\n2 - Signature verification\n3 - Exit")
        choose = input()
        if choose=='1':
            p = 57896044620753384133869303843568937902752767818974600847634902975134129543643
            q = 28948022310376692066934651921784468951377218528270520403696863131129758387393
            A = 1
            B = 52259530098387149819562511889780651425271270942919542722038553712464420235875
            x_P = 14539175448068301073584752148116082765715462525899666138074034449285211025933 
            y_P = 8328801466633898282311029798556417767141491055036399348346324804478619400451
            d = 28407275951127750820022741442645053622111847181874485046255279057194055286330
            if os.path.exists(file_name):
                GOST_34_10_2018_sender(p,q,A,B,x_P,y_P,d,file_name, decode_file)
            else:
                print("The file does not exist")
        elif choose=='2':
            new_file_name = 'encoded'
            decode = decode_file.replace(".signed", "")
            file_name, file_extension = os.path.splitext(decode)
            new_file_name=new_file_name + file_extension
            GOST_34_10_2018_recipient(decode_file, new_file_name)
        elif choose=='3':
            break;
        else:
            print("Wrong choose!")
    return 0
main()
    